# Norm-supervised highway experiments on Colab

This notebook reproduces the experiment matrix from `scripts/run_experiments.sh`. It clones a selected Git ref, creates an isolated virtual environment, runs each configuration through `scripts/test_highway.py`, and writes CSV results directly to Google Drive.

Before running, ensure the selected Git ref contains all required models and environment configs. Use a distinct `RUN_NAME` when running multiple Colab sessions in parallel.

In [ ]:
# Mount Google Drive. Colab will prompt you to authorize access.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Experiment parameters (edit this cell before running).
from datetime import datetime, timezone

REPO_URL = "https://github.com/phbui/norm-supervised-highway.git"  #@param {type:"string"}
REPO_REF = "main"  #@param {type:"string"}

NUM_EPISODES = 1000  #@param {type:"integer"}
ENVIRONMENTS = ["3L30V"]  # Valid: 2L10V, 3L30V, 4L40V, 4L20V, 6L50V
FORCE_WRITE = True  #@param {type:"boolean"}
FILE_PREFIX = "3L30V"  # Preserves the naming used by run_experiments.sh

# Each tuple is: (profile, method, value, enforce_filter).
# value must be None except for adaptive/fixed methods.
EXPERIMENTS = [
    ("right_lane", "nop", None, False),
    ("right_lane", "nop", None, True),
    ("right_lane", "naive", None, False),
    ("right_lane", "naive", None, True),
    ("right_lane", "adaptive", "0.05", False),
    ("right_lane", "adaptive", "0.05", True),
    ("right_lane", "fixed", "1.00", False),
    ("right_lane", "fixed", "1.00", True),
    ("right_lane", "projection", None, False),
    ("right_lane", "projection", None, True),
    ("right_lane", "adaptive", "0.0316", True),
    ("right_lane", "adaptive", "0.3162", True),
    ("right_lane", "adaptive", "3.1623", True),
    ("right_lane", "adaptive", "10.000", True),
]

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")  #@param {type:"string"}
DRIVE_RESULTS_ROOT = "/content/drive/MyDrive/aaai2027/norm-supervised-highway"  #@param {type:"string"}

In [ ]:
# Clone the requested ref and create the virtual environment.
import os
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path("/content/norm-supervised-highway")
VENV_DIR = Path("/content/venvs/norm-supervised-highway")
ENV_MODEL_FILES = {
    "2L10V": "models/4_lanes_20_vehicles.zip",
    "3L30V": "models/3_lanes_30_vehicles.zip",
    "4L40V": "models/3_lanes_30_vehicles.zip",
    "4L20V": "models/4_lanes_20_vehicles.zip",
    "6L50V": "models/4_lanes_20_vehicles.zip",
}

# Headless Colab rendering for pygame / highway-env.
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OFFSCREEN_RENDERING"] = "1"

for path in (PROJECT_DIR, VENV_DIR):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", REPO_REF], check=True)
subprocess.run(["python3", "-m", "venv", "--system-site-packages", str(VENV_DIR)], check=True)

VENV_PYTHON = VENV_DIR / "bin/python"
VENV_PIP = VENV_DIR / "bin/pip"
subprocess.run([str(VENV_PYTHON), "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([str(VENV_PIP), "install", "-e", str(PROJECT_DIR)], check=True)

entry_point = PROJECT_DIR / "scripts/test_highway.py"
if not entry_point.is_file():
    raise FileNotFoundError(f"Selected Git ref does not contain {entry_point}")

# Fail early if a selected env's model is missing or still a broken host symlink.
missing_models = []
for env_name in ENVIRONMENTS:
    if env_name not in ENV_MODEL_FILES:
        raise ValueError(f"Unknown environment {env_name!r}. Valid: {sorted(ENV_MODEL_FILES)}")
    model_path = PROJECT_DIR / ENV_MODEL_FILES[env_name]
    if model_path.is_symlink() and not model_path.exists():
        missing_models.append(f"{model_path} (broken symlink)")
    elif not model_path.is_file() or model_path.stat().st_size == 0:
        missing_models.append(str(model_path))
if missing_models:
    raise FileNotFoundError(
        "Commit real model zip files to the selected Git ref before running Colab:\n  - "
        + "\n  - ".join(missing_models)
    )

print(f"Project: {PROJECT_DIR}")
print(f"Python:  {VENV_PYTHON}")

In [ ]:
# Run the experiment matrix. CSV files and a complete log are written to Drive.
import json
import os

RUN_DIR = Path(DRIVE_RESULTS_ROOT).expanduser() / RUN_NAME
RESULTS_DIR = RUN_DIR / "results"
RUN_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "repo_url": REPO_URL,
    "repo_ref": REPO_REF,
    "num_episodes": NUM_EPISODES,
    "environments": ENVIRONMENTS,
    "force_write": FORCE_WRITE,
    "file_prefix": FILE_PREFIX,
    "experiments": EXPERIMENTS,
}
(RUN_DIR / "run_parameters.json").write_text(json.dumps(manifest, indent=2))

def run_and_tee(command, log_handle):
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_handle.write(line)
        log_handle.flush()
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

with (RUN_DIR / "run.log").open("a", buffering=1) as log:
    for env_name in ENVIRONMENTS:
        for profile, method, value, filter_enabled in EXPERIMENTS:
            suffix = f"{method}_{'filtered' if filter_enabled else 'unfiltered'}"
            output_dir = RESULTS_DIR / env_name / profile / suffix
            output_dir.mkdir(parents=True, exist_ok=True)
            value_suffix = f"_{value}" if method in {"adaptive", "fixed"} else ""
            output_path = output_dir / f"{FILE_PREFIX}_{env_name}{value_suffix}.csv"

            if output_path.exists() and not FORCE_WRITE:
                message = f"Skipping existing result: {output_path}\n"
                print(message, end="")
                log.write(message)
                continue

            command = [
                str(VENV_PYTHON), str(PROJECT_DIR / "scripts/test_highway.py"),
                "--profile", profile,
                "--method", method,
                "--episodes", str(NUM_EPISODES),
                "--env", env_name,
                "--output", str(output_path),
            ]
            if value is not None:
                command += ["--value", str(value)]
            if filter_enabled:
                command.append("--filter")
            run_and_tee(command, log)

print(f"Results saved to: {RUN_DIR}")

In [ ]:
# Verify the artifacts persisted to Drive.
for artifact in sorted(path for path in RUN_DIR.rglob("*") if path.is_file()):
    print(f"{artifact.relative_to(RUN_DIR)}  ({artifact.stat().st_size:,} bytes)")